# Processing CESM2-LENS HIST-SSP370 ensemble member data

### Set up
#### Packages

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from Processing_functions import FixLongitude, FixTime, FixGrid, InterPlevels
xr.set_options(display_expand_data=False);
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import random
import os

#### Filepaths & name variables

In [ ]:
LELM = True # True: processing LENS2-smbb, False: processing lessmelt-smbb

## File name
if LELM:
    cesm2LE_HIST = 'b.e21.BHISTsmbb.f09_g17'
    cesm2LE_SSP = 'b.e21.BSSP370smbb.f09_g17'
    cesm2LE_outname = 'b.e21.BHISTsmbb.f09_g17.LE-smbb'
    # CHANGE
    yr_start = 1950
    yr_end = 2024
else:
    cesm2LE_HIST = 'b.e21.BHISTsmbb.f09_g17.CMIP6-historical'
    cesm2LE_outname = 'b.e21.BHISTsmbb.f09_g17.LM-smbb'
    # DON'T CHANGE
    yr_start = 1999
    yr_end = 2014

## Filepaths
comp = 'atm'
freq = 0 # 0: monthly, 1: daily
var_ind = 0
path_to_arch = "/glade/campaign/cgd/cesm/CESM2-LE/"+comp+"/proc/tseries/month_1/" if LELM else "/glade/campaign/cgd/ppc/cesm2_tuned_albedo/"
path_to_tseries = "/"+comp+"/proc/tseries/month_1/"

# ATM
# 0,     1     
# TREFHT, RESTOM

# ICE
# 0,    1 
# aice, hi

# OCN
# 0
# MOC

# Variables
var_list = {'atm': ['TREFHT','RESTOM'],
            'ice': ['aice','hi'],
            'ocn': ['MOC']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

# Extensions
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
mod_com = {'atm': 'cam',
           'ice': 'cice',
           'ocn': 'pop'}
time_path = {'atm': ['month_1'],
                'ice': ['month_1','day_1'],
                'ocn': ['month_1']}
vert_lev = {'atm': [False,False],
            'ice': [False,False],
            'ocn': [False]}
yr_extn =  {'out': [".195001-202412."]}

path_to_outdata = '/glade/work/glydia/processed_CESM2_LENS_data/longer_recent/'
plot_levels = [300,500,850,925]

In [ ]:
cluster = PBSCluster(cores    = 1,
                     memory   = '25GiB',
                     queue    = 'casper',
                     walltime = '04:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='HIST-SSP370smbb_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [ ]:
client

In [ ]:
## Chunking variables
time_ch = 600
chunks = {
    'atm': {'time': time_ch, 'lat': 96, 'lon': 144, 'lev': -1},
    'ice': {'time': time_ch, 'nj': 192, 'ni': 160},
    'ocn': {'time': time_ch, 'nlat': 64, 'nlon': 96, 'z_t': 5}
}

### Load & modify data
#### Control data

In [ ]:
ens_codes = np.array(['LE2-1011.001', 'LE2-1031.002', 'LE2-1051.003', 'LE2-1071.004',
       'LE2-1091.005', 'LE2-1111.006', 'LE2-1131.007', 'LE2-1151.008',
       'LE2-1171.009', 'LE2-1191.010', 'LE2-1231.011', 'LE2-1231.012',
       'LE2-1231.013', 'LE2-1231.014', 'LE2-1231.015', 'LE2-1231.016',
       'LE2-1231.017', 'LE2-1231.018', 'LE2-1231.019', 'LE2-1231.020',
       'LE2-1251.011', 'LE2-1251.012', 'LE2-1251.013', 'LE2-1251.014',
       'LE2-1251.015', 'LE2-1251.016', 'LE2-1251.017', 'LE2-1251.018',
       'LE2-1251.019', 'LE2-1251.020', 'LE2-1281.011', 'LE2-1281.012',
       'LE2-1281.013', 'LE2-1281.014', 'LE2-1281.015', 'LE2-1281.016',
       'LE2-1281.017', 'LE2-1281.018', 'LE2-1281.019', 'LE2-1281.020',
       'LE2-1301.011', 'LE2-1301.012', 'LE2-1301.013', 'LE2-1301.014',
       'LE2-1301.015', 'LE2-1301.016', 'LE2-1301.017', 'LE2-1301.018',
       'LE2-1301.019', 'LE2-1301.020']) if LELM else np.array([str(i).zfill(3) for i in range(118,123)])

startyrs_hist = np.arange(1950,2011,10) if LELM else np.array([1990])
if LELM:
    endyrs_hist = np.arange(1959,2010,10)
    endyrs_hist = np.append(endyrs_hist,2014)
else:
    endyrs_hist = np.array([2014])

yrs_hist = []
for i in range(0,len(startyrs_hist)):
    yrs_hist.append('.'+str(startyrs_hist[i])+'01-'+str(endyrs_hist[i])+'12.')

if LELM:
    startyrs_ssp = np.array([2015])
    endyrs_ssp = np.array([2024])
    # endyrs_ssp = np.array([2064])
    yrs_ssp = ['.'+str(startyrs_ssp[0])+'01-'+str(endyrs_ssp[0])+'12.']

yr_range = np.arange(yr_start,yr_end+2)
yr_range = [str(i) for i in yr_range]

In [ ]:
not_vert = not vert_lev[comp][var_ind]
slice_numbers = np.arange(1,51) if LELM else np.arange(1,6)

In [ ]:
%%time

# If 2D and not RESTOM
if var != 'RESTOM' and not_vert:
    num = 0
    for e_code in ens_codes:
        if LELM:
            path_list = [path_to_arch+var+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr+'nc' for yr in yrs_hist]
            path_list.append(path_to_arch+var+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yrs_ssp[0]+'nc')
        else:
            path_list = [path_to_arch+cesm2LE_HIST+'.'+e_code+path_to_tseries+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yrs_hist[0]+'nc']
        

        print(path_list[0])

        ds = xr.open_mfdataset(path_list, chunks=chunks[comp], parallel=True) if LELM else xr.open_mfdataset(path_list[0], chunks=chunks[comp])
        dsv = ds[var]
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr_sl = yr_range[i]
                endyr_sl = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) 
                print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
                
                fixedtime_data = FixTime(ann_slice)
                
                print('   fixed time')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ice','gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                elif comp == 'ocn':
                    if var == 'TEMP':
                        fixedtime_data = fixedtime_data.sel(z_t=0,method='nearest')
                        fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ocn','gx1v7')
                        processed_list.append(fixedgrid_data)
                        print('   fixed CICE grid')
                    else:
                        processed_list.append(fixedtime_data)
        
                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                    processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
        
        if freq == 0 and not_vert:
            processed_comp = dask.compute(*processed_list)
            print('computed list')
            
            processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
            print('concatenated data')
    
            processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
            
            processed_out.to_netcdf(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)
                                    +h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                        format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
            print('wrote data to disk')
    
        num = num+1

In [ ]:
%%time

# If 3D variable
if var != 'RESTOM' and vert_lev[comp][var_ind]:
    num = 0
    for e_code in ens_codes:
        path_list = [path_to_arch+var+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr+'nc' for yr in yrs_hist]
        path_list.append(path_to_arch+var+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yrs_ssp[0]+'nc')

        print(path_list[0])

        ds = xr.open_mfdataset(path_list, chunks=chunks[comp], parallel=True)
        dsv = ds

        path_listP = [path_to_arch+'PS'+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr+'nc' for yr in yrs_hist]
        path_listP.append(path_to_arch+'PS'+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yrs_ssp[0]+'nc')
        dsp = xr.open_mfdataset(path_listP,chunks=chunks[comp])
            
        dsv['PS'] = dsp['PS']
        #del ds
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # Selection of final time
            final_time = pd.date_range(str(1950+i)+'-01-01',str(1950+i)+'-12-01',freq='MS')
                
            # Selection of data    
            startyr_sl = yr_range[i]
            endyr_sl = yr_range[i+1]
            ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
            print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
            
            ann_slice = FixTime(ann_slice)
            ann_slice = ann_slice.assign_coords(time=final_time)
            print('   fixed time')

            ann_slice = FixLongitude(ann_slice, False)
            print('   fixed longitude')
            
            ann_slice = InterPlevels(ann_slice, var)
            print('   interpolated p-levels')
            processed_list.append(ann_slice)

        
        processed_out = xr.concat(processed_list,dim='time').chunk({'time':111})
        processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
        slice_list.append(processed_out)  
        print('concatenated slice '+str(num))
        num = num+1
        
        for i in range(0,len(slice_list)):
            if i == 0:
                slice_list[i].to_zarr(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr', 
                                group=var)
                print('saved initial zarr store')
            else:
                print('saving slice '+str(i+1))
                slice_list[i].to_zarr(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr', 
                                        append_dim='slice', mode='a-',group=var)
        print('wrote data to disk')
    
       

In [ ]:
%%time
if var == 'RESTOM':
    num = 0
    for e_code in ens_codes:
        if LELM:
            path_list_flnt = [path_to_arch+'FLNT'+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr+'nc' for yr in yrs_hist]
            path_list_flnt.append(path_to_arch+'FLNT'+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yrs_ssp[0]+'nc')
                
            path_list_fsnt = [path_to_arch+'FSNT'+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr+'nc' for yr in yrs_hist]
            path_list_fsnt.append(path_to_arch+'FSNT'+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yrs_ssp[0]+'nc')
        else:
            path_list_flnt = [path_to_arch+cesm2LE_HIST+'.'+e_code+path_to_tseries+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yrs_hist[0]+'nc']
            path_list_fsnt = [path_to_arch+cesm2LE_HIST+'.'+e_code+path_to_tseries+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yrs_hist[0]+'nc']
        

        print(path_list_fsnt[0])

        if LELM:
            ds_fsnt = xr.open_mfdataset(path_list_fsnt, chunks=chunks[comp], parallel=True)
            ds_flnt = xr.open_mfdataset(path_list_flnt, chunks=chunks[comp], parallel=True)
        else:
            ds_fsnt = xr.open_mfdataset(path_list_fsnt[0], chunks=chunks[comp])
            ds_flnt = xr.open_mfdataset(path_list_flnt[0], chunks=chunks[comp])        
    
        ds_fsnt = ds_fsnt['FSNT']
        ds_flnt = ds_flnt['FLNT']

        dsv = ds_fsnt-ds_flnt
        dsv = dsv.rename(var)
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr_sl = yr_range[i]
                endyr_sl = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) 
                print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
                
                fixedtime_data = FixTime(ann_slice)
                print('   fixed time')
        
                fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                processed_list.append(fixedgrid_data)
                print('   fixed longitude')
        
        processed_comp = dask.compute(*processed_list)
        print('computed list')
        
        processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
        print('concatenated data')

        processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
        
        processed_out.to_netcdf(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)
                                +h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                    format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
        print('wrote data to disk')
    
        num = num+1

In [ ]:
client.shutdown()